# Silent Signal — kiểm tra RTMPose-L WholeBody trên Google Colab

Notebook này kiểm tra end-to-end nhánh trích xuất pose sau khi ASL Citizen đã được chuẩn bị: GPU, môi trường OpenMMLab cố định, checkpoint RTMDet-M và RTMPose-L 384×288, SHA-256, unit test, trích xuất một clip, kiểm tra cache NPZ 133 keypoint, trực quan hóa và resume.

Pipeline dùng đường dẫn model/config tường minh, không dùng alias `wholebody`. Checkpoint được giữ trên Google Drive; `/content` và môi trường Python sẽ mất khi Colab reset. Chạy các cell từ trên xuống. Các thao tác tốn thời gian đều có cờ bật/tắt.

Trước khi bắt đầu: chọn **Runtime → Change runtime type → T4 GPU** (hoặc L4/A100), bảo đảm code pose mới đã được push lên nhánh Git được chọn, và hoàn tất `00_asl_citizen_colab_preparation.ipynb` trong cùng runtime nếu dataset đang ở `/content`.

Tài liệu chính thức: [MMPose installation](https://mmpose.readthedocs.io/en/latest/installation.html), [MMPose compatibility](https://mmpose.readthedocs.io/en/latest/faq.html), [MMCV 2.1.0 CUDA 12.1 / PyTorch 2.1 wheels](https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stillthethrone/silent-signal/blob/dev/notebooks/03_rtmpose_wholebody_colab_check.ipynb)


## 1. Cấu hình và mount Google Drive

Giá trị mặc định nối với output của notebook chuẩn bị ASL Citizen. Nếu dataset được lưu ở Drive hoặc vị trí khác, chỉ sửa `DATASET_ROOT`. Để kiểm tra clip khó cụ thể, điền `SMOKE_SAMPLE_ID`; nếu để trống notebook chọn một clip train theo thứ tự ổn định.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_GIT_URL = 'https://github.com/stillthethrone/silent-signal.git'
PROJECT_GIT_REF = 'dev'  # @param {type:'string'}
PROJECT_ROOT = Path('/content/silent-signal')
MMPOSE_ROOT = Path('/content/mmpose-v1.3.2')
POSE_ENV_ROOT = Path('/content/pose-env')

DATASET_ROOT = Path('/content/ASL_Citizen')  # @param {type:'string'}
PREPARATION_ROOT = Path('/content/drive/MyDrive/silent-signal-results/asl_citizen')
MANIFEST_PATH = PREPARATION_ROOT / 'manifests/asl_citizen.parquet'
POSE_RESULT_ROOT = PREPARATION_ROOT / 'pose/rtmpose_l_coco_wholebody_384x288'
MODEL_ROOT = Path('/content/drive/MyDrive/silent-signal-models/openmmlab')

RUN_UNIT_TESTS = True  # @param {type:'boolean'}
DOWNLOAD_MODELS = True  # @param {type:'boolean'}
RUN_SMOKE = True  # @param {type:'boolean'}
OVERWRITE_SMOKE = False  # @param {type:'boolean'}
RUN_RESUME_CHECK = True  # @param {type:'boolean'}
SMOKE_SAMPLE_ID = ''  # @param {type:'string'}
SMOKE_SPLIT = 'train'  # @param ['train', 'validation', 'test']
SMOKE_LIMIT = 1  # @param {type:'integer'}

RUN_PILOT = False  # @param {type:'boolean'}
PILOT_SPLIT = 'train'  # @param ['train', 'validation', 'test']
PILOT_LIMIT = 20  # @param {type:'integer'}

if SMOKE_LIMIT < 1 or PILOT_LIMIT < 1:
    raise ValueError('SMOKE_LIMIT và PILOT_LIMIT phải lớn hơn 0.')
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
POSE_RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET_ROOT)
print('Manifest:', MANIFEST_PATH)
print('Pose output:', POSE_RESULT_ROOT)


## 2. Clone hoặc cập nhật source

Notebook chỉ fast-forward khi thư mục source sạch và đúng remote/branch. Với repository private, xác thực GitHub bằng cơ chế riêng của bạn; không ghi token vào notebook.


In [ ]:
import os
import subprocess

def run(command, **kwargs):
    print('+', ' '.join(str(part) for part in command))
    return subprocess.run(command, check=True, **kwargs)

def git_output(root, *arguments):
    return subprocess.check_output(
        ['git', '-C', str(root), *arguments], text=True
    ).strip()

if not PROJECT_ROOT.exists():
    run([
        'git', 'clone', '--branch', PROJECT_GIT_REF, '--depth', '1',
        PROJECT_GIT_URL, str(PROJECT_ROOT),
    ])
else:
    if not (PROJECT_ROOT / '.git').is_dir():
        raise RuntimeError('PROJECT_ROOT tồn tại nhưng không phải Git repository.')
    if git_output(PROJECT_ROOT, 'remote', 'get-url', 'origin') != PROJECT_GIT_URL:
        raise RuntimeError('Repository local có origin khác PROJECT_GIT_URL.')
    if git_output(PROJECT_ROOT, 'branch', '--show-current') != PROJECT_GIT_REF:
        raise RuntimeError('Repository local đang ở nhánh khác PROJECT_GIT_REF.')
    if git_output(PROJECT_ROOT, 'status', '--porcelain'):
        raise RuntimeError('Repository local có thay đổi chưa commit; không tự động pull.')
    run([
        'git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only',
        'origin', PROJECT_GIT_REF,
    ])

required_project_files = [
    PROJECT_ROOT / 'configs/pose/rtmpose.yaml',
    PROJECT_ROOT / 'src/silent_signal/pose/rtmpose.py',
    PROJECT_ROOT / 'src/silent_signal/cli/extract_pose.py',
]
missing_project_files = [str(path) for path in required_project_files if not path.is_file()]
if missing_project_files:
    raise RuntimeError(
        'Nhánh đã tải chưa có pipeline pose mới: ' + ', '.join(missing_project_files)
    )
PROJECT_COMMIT = git_output(PROJECT_ROOT, 'rev-parse', 'HEAD')
os.chdir(PROJECT_ROOT)
print('Project commit:', PROJECT_COMMIT)


## 3. Kiểm tra GPU Colab

Dòng `nvidia-smi` phải hiển thị GPU. Nếu không có, bật GPU trong Runtime rồi chạy lại từ đầu.


In [ ]:
import shutil

run(['nvidia-smi'])
disk = shutil.disk_usage('/content')
print(f'/content còn trống: {disk.free / 1024**3:.2f} GiB')


## 4. Tạo môi trường OpenMMLab tái lập được

Môi trường riêng dùng Python 3.11, NumPy 1.26.4, PyTorch 2.1.0 CUDA 12.1, MMCV 2.1.0, MMEngine 0.10.7, MMDetection 3.2.0 và MMPose 1.3.2. Việc này tránh phụ thuộc vào phiên bản Python/PyTorch mặc định thay đổi theo Colab. Cài MMCV sau PyTorch; không cài đồng thời `mmcv-lite` hoặc `mmcv-full`. Cell có thể mất vài phút.


In [ ]:
import sys

run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'])
run(['uv', 'python', 'install', '3.11'])
if not (POSE_ENV_ROOT / 'bin/python').is_file():
    run(['uv', 'venv', str(POSE_ENV_ROOT), '--python', '3.11', '--seed'])

POSE_PY = str(POSE_ENV_ROOT / 'bin/python')
POSE_CLI = str(POSE_ENV_ROOT / 'bin/ss-extract-pose')

run([POSE_PY, '-m', 'pip', 'install', '--upgrade', 'pip', 'wheel'])
run([POSE_PY, '-m', 'pip', 'install', 'numpy==1.26.4'])
run([
    POSE_PY, '-m', 'pip', 'install',
    'torch==2.1.0', 'torchvision==0.16.0',
    '--index-url', 'https://download.pytorch.org/whl/cu121',
    '--extra-index-url', 'https://pypi.org/simple',
])
run([POSE_PY, '-m', 'pip', 'install', 'mmengine==0.10.7'])
run([
    POSE_PY, '-m', 'pip', 'install', 'mmcv==2.1.0',
    '-f', 'https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html',
])
run([POSE_PY, '-m', 'pip', 'install', 'mmdet==3.2.0'])

if not MMPOSE_ROOT.exists():
    run([
        'git', 'clone', '--depth', '1', '--branch', 'v1.3.2',
        'https://github.com/open-mmlab/mmpose.git', str(MMPOSE_ROOT),
    ])
elif not (MMPOSE_ROOT / '.git').is_dir():
    raise RuntimeError('MMPOSE_ROOT tồn tại nhưng không phải source MMPose.')

MMPOSE_COMMIT = git_output(MMPOSE_ROOT, 'rev-parse', 'HEAD')
run([POSE_PY, '-m', 'pip', 'install', '-e', str(MMPOSE_ROOT)])
run([POSE_PY, '-m', 'pip', 'install', '-e', f'{PROJECT_ROOT}[dev]'])
print('MMPose commit:', MMPOSE_COMMIT)


## 5. Xác nhận version, CUDA và MMCV ops

Import `mmcv.ops` là kiểm tra quan trọng. Lỗi `mmcv._ext` hoặc `mmcv.ops` thường cho biết wheel MMCV không khớp PyTorch/CUDA.


In [ ]:
environment_check = r'''
import cv2
import mmcv
import mmengine
import mmdet
import mmpose
import numpy as np
import torch
from mmcv.ops import nms

assert torch.cuda.is_available(), 'CUDA không khả dụng trong pose-env.'
assert torch.ones(1, device='cuda:0').is_cuda
print('Python   : OK')
print('PyTorch  :', torch.__version__)
print('CUDA     :', torch.version.cuda)
print('GPU      :', torch.cuda.get_device_name(0))
print('NumPy    :', np.__version__)
print('OpenCV   :', cv2.__version__)
print('MMCV     :', mmcv.__version__)
print('MMEngine :', mmengine.__version__)
print('MMDet    :', mmdet.__version__)
print('MMPose   :', mmpose.__version__)
print('mmcv.ops : OK')
'''
run([POSE_PY, '-c', environment_check])


## 6. Chạy test project

Test này không thay thế smoke test GPU, nhưng xác nhận layout 75 node, cache, cấu hình, signer selection và CLI vẫn đúng sau khi clone lên Colab. Với trạng thái code lúc notebook được tạo, kỳ vọng `98 passed`.


In [ ]:
if RUN_UNIT_TESTS:
    run([POSE_PY, '-m', 'pytest', '-q'], cwd=PROJECT_ROOT)
else:
    print('Bỏ qua unit tests theo cấu hình.')


## 7. Tải checkpoint chính thức vào Drive

Download dùng file `.part` và `wget -c` để có thể tiếp tục nếu mạng ngắt. Sau khi tải xong mới đổi tên thành `.pth`. Nếu file đích đã tồn tại, cell không tải lại; bước verify tiếp theo vẫn tính SHA-256 toàn bộ file.


In [ ]:
POSE_URL = (
    'https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/'
    'rtmpose-l_simcc-coco-wholebody_pt-aic-coco_270e-384x288-'
    'eaeb96c8_20230125.pth'
)
DET_URL = (
    'https://download.openmmlab.com/mmpose/v1/projects/rtmpose/'
    'rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth'
)
POSE_CHECKPOINT = MODEL_ROOT / 'rtmpose-l-wholebody-384x288.pth'
DET_CHECKPOINT = MODEL_ROOT / 'rtmdet-m-person.pth'

def download_checkpoint(url, destination):
    if destination.is_file():
        print('Đã tồn tại:', destination)
        return
    partial = Path(str(destination) + '.part')
    run(['wget', '-c', url, '-O', str(partial)])
    if not partial.is_file() or partial.stat().st_size == 0:
        raise RuntimeError(f'Download rỗng: {partial}')
    partial.replace(destination)
    print('Đã tải:', destination)

if DOWNLOAD_MODELS:
    download_checkpoint(POSE_URL, POSE_CHECKPOINT)
    download_checkpoint(DET_URL, DET_CHECKPOINT)
else:
    print('Bỏ qua download; sẽ dùng checkpoint đã có trên Drive.')


## 8. Kiểm tra đầu vào và khai báo biến môi trường

`MANIFEST_PATH` là manifest đã kiểm tra từ notebook chuẩn bị dữ liệu. Mọi `video_path` trong manifest phải là đường dẫn tương đối tồn tại dưới `DATASET_ROOT`.


In [ ]:
import json

SOURCE_POSE_CONFIG = PROJECT_ROOT / 'configs/pose/rtmpose.yaml'
POSE_MODEL_CONFIG = (
    MMPOSE_ROOT / 'configs/wholebody_2d_keypoint/rtmpose/coco-wholebody/'
    'rtmpose-l_8xb32-270e_coco-wholebody-384x288.py'
)
DET_MODEL_CONFIG = (
    MMPOSE_ROOT / 'demo/mmdetection_cfg/rtmdet_m_640-8xb32_coco-person.py'
)

required_inputs = [
    DATASET_ROOT, MANIFEST_PATH, SOURCE_POSE_CONFIG,
    POSE_MODEL_CONFIG, DET_MODEL_CONFIG,
    POSE_CHECKPOINT, DET_CHECKPOINT,
]
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(
        'Thiếu đầu vào. Kiểm tra DATASET_ROOT/MANIFEST/checkpoint: '
        + ', '.join(missing_inputs)
    )

status_path = PREPARATION_ROOT / 'reports/preparation_status.json'
if status_path.is_file():
    preparation_status = json.loads(status_path.read_text(encoding='utf-8'))
    print('Preparation status:')
    print(json.dumps(preparation_status, ensure_ascii=False, indent=2))
else:
    print('Cảnh báo: không tìm thấy preparation_status.json; tiếp tục với manifest hiện có.')

os.environ['ASL_CITIZEN_ROOT'] = str(DATASET_ROOT)
os.environ['MMPOSE_ROOT'] = str(MMPOSE_ROOT)
os.environ['RTMPOSE_L_WHOLEBODY_CHECKPOINT'] = str(POSE_CHECKPOINT)
os.environ['RTMDET_M_PERSON_CHECKPOINT'] = str(DET_CHECKPOINT)
print('Tất cả đầu vào bắt buộc đã tồn tại.')


## 9. Tính SHA-256, tạo config đã khóa và verify lại

Lần verify đầu tính hash thực tế. Notebook ghi hai hash vào một bản sao YAML trên Drive, rồi verify lần hai. Source config trong Git không bị sửa. Lock cuối chứa hash config/checkpoint, phiên bản package, CUDA và GPU.


In [ ]:
import yaml

PROVENANCE_ROOT = POSE_RESULT_ROOT / 'provenance'
PROVENANCE_ROOT.mkdir(parents=True, exist_ok=True)
UNPINNED_LOCK = PROVENANCE_ROOT / 'unverified-hashes.lock.json'
PINNED_CONFIG = PROVENANCE_ROOT / 'rtmpose-colab-pinned.yaml'
PINNED_LOCK = PROVENANCE_ROOT / 'rtmpose-colab-pinned.lock.json'

run([
    POSE_CLI, 'verify', '--config', str(SOURCE_POSE_CONFIG),
    '--write-lock', str(UNPINNED_LOCK),
])
unverified = json.loads(UNPINNED_LOCK.read_text(encoding='utf-8'))
pose_sha256 = unverified['pose_model']['checkpoint_sha256']
det_sha256 = unverified['detector']['checkpoint_sha256']

config_payload = yaml.safe_load(SOURCE_POSE_CONFIG.read_text(encoding='utf-8'))
config_payload['extractor']['pose_model']['checkpoint_sha256'] = pose_sha256
config_payload['extractor']['detector']['checkpoint_sha256'] = det_sha256
PINNED_CONFIG.write_text(
    yaml.safe_dump(config_payload, sort_keys=False, allow_unicode=True),
    encoding='utf-8',
)

run([
    POSE_CLI, 'verify', '--config', str(PINNED_CONFIG),
    '--write-lock', str(PINNED_LOCK),
])
verified = json.loads(PINNED_LOCK.read_text(encoding='utf-8'))
print('RTMPose SHA-256:', pose_sha256)
print('RTMDet  SHA-256:', det_sha256)
print('Pinned config:', PINNED_CONFIG)
print('Pinned lock:', PINNED_LOCK)


## 10. Smoke test một clip

Mặc định chọn một clip train. Nếu `SMOKE_SAMPLE_ID` có giá trị, notebook chỉ chọn sample đó và không lọc split. Cache/report được lưu trên Drive. Nếu cache cũ không cùng fingerprint, pipeline sẽ từ chối ghi đè; chỉ bật `OVERWRITE_SMOKE=True` sau khi đã kiểm tra đúng thư mục smoke.


In [ ]:
import hashlib

if SMOKE_SAMPLE_ID.strip():
    sample_digest = hashlib.sha256(SMOKE_SAMPLE_ID.strip().encode('utf-8')).hexdigest()[:12]
    smoke_run_name = f'sample_{sample_digest}'
else:
    smoke_run_name = f'{SMOKE_SPLIT}_first_{SMOKE_LIMIT}'
SMOKE_ROOT = POSE_RESULT_ROOT / 'smoke' / smoke_run_name
SMOKE_OUTPUT_ROOT = SMOKE_ROOT / 'raw'
SMOKE_REPORT = SMOKE_ROOT / 'report.json'
SMOKE_ROOT.mkdir(parents=True, exist_ok=True)

smoke_command = [
    POSE_CLI, 'extract',
    '--config', str(PINNED_CONFIG),
    '--manifest', str(MANIFEST_PATH),
    '--dataset-root', str(DATASET_ROOT),
    '--output-root', str(SMOKE_OUTPUT_ROOT),
    '--report', str(SMOKE_REPORT),
    '--device', 'cuda:0',
    '--progress-every', '1',
]
if SMOKE_SAMPLE_ID.strip():
    smoke_command.extend(['--sample-id', SMOKE_SAMPLE_ID.strip()])
else:
    smoke_command.extend([
        '--split', SMOKE_SPLIT, '--limit', str(SMOKE_LIMIT),
    ])
if OVERWRITE_SMOKE:
    smoke_command.append('--overwrite')

if RUN_SMOKE:
    smoke_result = subprocess.run(smoke_command)
    if not SMOKE_REPORT.is_file():
        raise RuntimeError('Extractor không tạo smoke report.')
    smoke_report = json.loads(SMOKE_REPORT.read_text(encoding='utf-8'))
    print(json.dumps(smoke_report, ensure_ascii=False, indent=2))
    if smoke_result.returncode != 0:
        raise RuntimeError('Smoke extraction thất bại; xem failures trong report.')
    assert smoke_report['selected'] > 0
    assert smoke_report['failed'] == 0
    assert (
        smoke_report['extracted'] + smoke_report['resumed']
        == smoke_report['selected']
    )
else:
    print('Bỏ qua smoke extraction theo cấu hình.')


## 11. Kiểm tra cấu trúc và chất lượng số học của cache

Các điều kiện cứng: đúng 133 điểm, shape nhất quán, không NaN/Inf, timestamp không giảm và bbox nằm trong frame ở các frame phát hiện signer. Detection rate và confidence tay là chỉ số chẩn đoán, chưa phải ngưỡng loại dữ liệu cố định.


In [ ]:
import numpy as np

cache_files = sorted(SMOKE_OUTPUT_ROOT.rglob('*.npz'))
if not cache_files:
    raise FileNotFoundError(f'Không có cache NPZ dưới {SMOKE_OUTPUT_ROOT}')
if SMOKE_LIMIT == 1 or SMOKE_SAMPLE_ID.strip():
    assert len(cache_files) == 1, cache_files
CACHE_PATH = cache_files[0]

with np.load(CACHE_PATH, allow_pickle=False) as archive:
    frame_indices = archive['frame_indices'].copy()
    timestamps = archive['timestamps_seconds'].copy()
    keypoints = archive['keypoints_xy'].copy()
    scores = archive['keypoint_scores'].copy()
    bboxes = archive['bboxes_xyxy'].copy()
    bbox_scores = archive['bbox_scores'].copy()
    detected = archive['person_detected'].copy()
    envelope = json.loads(archive['metadata_json'].tobytes().decode('utf-8'))

frame_count = len(frame_indices)
assert keypoints.shape == (frame_count, 133, 2)
assert scores.shape == (frame_count, 133)
assert bboxes.shape == (frame_count, 4)
assert bbox_scores.shape == (frame_count,)
assert detected.shape == (frame_count,)
assert np.isfinite(keypoints).all()
assert np.isfinite(scores).all()
assert np.isfinite(bboxes).all()
assert np.isfinite(bbox_scores).all()
assert np.all(np.diff(timestamps) >= 0)

height, width = envelope['frame_size_hw']
if detected.any():
    valid_boxes = bboxes[detected]
    assert (valid_boxes[:, 0] >= 0).all()
    assert (valid_boxes[:, 1] >= 0).all()
    assert (valid_boxes[:, 2] <= width).all()
    assert (valid_boxes[:, 3] <= height).all()
    assert (valid_boxes[:, 2] >= valid_boxes[:, 0]).all()
    assert (valid_boxes[:, 3] >= valid_boxes[:, 1]).all()

def mean_score(start, end):
    return float(scores[detected, start:end].mean()) if detected.any() else float('nan')

quality = {
    'cache_path': str(CACHE_PATH),
    'sample_id': envelope['sample_id'],
    'source_video': envelope['source_video'],
    'frames': frame_count,
    'keypoints_shape': list(keypoints.shape),
    'person_detection_rate': float(detected.mean()),
    'missing_person_frames': int((~detected).sum()),
    'body_mean_score': mean_score(0, 17),
    'face_mean_score': mean_score(23, 91),
    'left_hand_mean_score': mean_score(91, 112),
    'right_hand_mean_score': mean_score(112, 133),
}
print(json.dumps(quality, ensure_ascii=False, indent=2))


## 12. Trực quan hóa trực tiếp từ cache của pipeline

Bốn frame phân bố theo thời gian giúp kiểm tra bbox, mặt và hai tay. Đây là dữ liệu chính pipeline đã lưu, không phải output từ alias hoặc một demo khác. Chỉ vẽ điểm có confidence từ `0.25`.


In [ ]:
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

valid_rows = np.flatnonzero(detected)
if len(valid_rows) == 0:
    raise RuntimeError('Không frame nào phát hiện signer; không thể trực quan hóa.')
number_to_show = min(4, len(valid_rows))
selected_rows = valid_rows[
    np.linspace(0, len(valid_rows) - 1, number_to_show).astype(int)
]
groups = {
    'body': (0, 17, 'cyan'),
    'face': (23, 91, 'orange'),
    'left hand': (91, 112, 'lime'),
    'right hand': (112, 133, 'magenta'),
}

capture = cv2.VideoCapture(envelope['source_video'])
if not capture.isOpened():
    raise RuntimeError(f"Không mở được video: {envelope['source_video']}")
fig, axes = plt.subplots(1, number_to_show, figsize=(5 * number_to_show, 5))
axes = np.atleast_1d(axes)
try:
    for axis, row in zip(axes, selected_rows):
        frame_number = int(frame_indices[row])
        capture.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ok, frame = capture.read()
        if not ok:
            raise RuntimeError(f'Không đọc được frame {frame_number}')
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        axis.imshow(frame)
        x1, y1, x2, y2 = bboxes[row]
        axis.add_patch(Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            fill=False, color='red', linewidth=2,
        ))
        for name, (start, end, color) in groups.items():
            visible = scores[row, start:end] >= 0.25
            points = keypoints[row, start:end][visible]
            if len(points):
                axis.scatter(
                    points[:, 0], points[:, 1], s=12, c=color, label=name
                )
        axis.set_title(f'Frame {frame_number}')
        axis.axis('off')
finally:
    capture.release()
axes[0].legend(loc='upper right')
plt.tight_layout()
plt.show()


## 13. Kiểm tra resume

Chạy lại cùng lệnh phải cho `extracted=0`, `resumed=selected`, `failed=0`. Tắt `OVERWRITE_SMOKE` khi kiểm tra resume.


In [ ]:
if RUN_RESUME_CHECK:
    if OVERWRITE_SMOKE:
        raise ValueError('Tắt OVERWRITE_SMOKE trước khi kiểm tra resume.')
    run(smoke_command)
    resume_report = json.loads(SMOKE_REPORT.read_text(encoding='utf-8'))
    print(json.dumps(resume_report, ensure_ascii=False, indent=2))
    assert resume_report['failed'] == 0
    assert resume_report['extracted'] == 0
    assert resume_report['resumed'] == resume_report['selected']
else:
    print('Bỏ qua resume check theo cấu hình.')


## 14. Pilot tùy chọn

Sau khi smoke test đạt, đặt `RUN_PILOT=True` và chạy 20 clip trước. Tăng dần lên 100 sau khi xem report và một số video khó. Pilot dùng output riêng và `--continue-on-error` để luôn ghi đủ danh sách lỗi.


In [ ]:
PILOT_ROOT = POSE_RESULT_ROOT / f'pilot_{PILOT_SPLIT}_{PILOT_LIMIT}'
PILOT_REPORT = PILOT_ROOT / 'report.json'
pilot_command = [
    POSE_CLI, 'extract',
    '--config', str(PINNED_CONFIG),
    '--manifest', str(MANIFEST_PATH),
    '--dataset-root', str(DATASET_ROOT),
    '--output-root', str(PILOT_ROOT / 'raw'),
    '--report', str(PILOT_REPORT),
    '--split', PILOT_SPLIT,
    '--limit', str(PILOT_LIMIT),
    '--device', 'cuda:0',
    '--continue-on-error',
    '--progress-every', '5',
]
if RUN_PILOT:
    pilot_result = subprocess.run(pilot_command)
    if not PILOT_REPORT.is_file():
        raise RuntimeError('Pilot không tạo report.')
    pilot_report = json.loads(PILOT_REPORT.read_text(encoding='utf-8'))
    print(json.dumps(pilot_report, ensure_ascii=False, indent=2))
    print('Pilot exit code:', pilot_result.returncode)
else:
    print('RUN_PILOT=False — chưa chạy pilot.')


## Tiêu chí đạt

- Environment cell in đúng version và `mmcv.ops: OK`.
- Unit tests pass.
- Verify config đã khóa SHA-256 thành công.
- Smoke report có `failed=0` và `extracted + resumed = selected`.
- Cache có shape `[T, 133, 2]`, không NaN/Inf và timestamp không giảm.
- Bbox đỏ bám đúng signer; keypoint tay/mặt không lệch nền hay đổi sang người khác.
- Resume report có `extracted=0`, `resumed=selected`.

Nếu runtime sau cấp loại GPU khác, fingerprint môi trường có thể thay đổi và cache cũ sẽ bị đánh dấu stale. Khi đó không ghi đè mù quáng: giữ output cũ, tạo output root mới hoặc chủ động bật overwrite sau khi xác nhận model/config/checkpoint vẫn đúng. Không nên chạy toàn bộ 83.399 clip trong một phiên Colab miễn phí; chạy pilot và shard theo GPU/runtime trước.
